In [1]:
from align_masks import align_masks
from collections import defaultdict
from matplotlib import pyplot as plt
from scipy.ndimage import shift
import numpy as np
import cv2
import os
import re


In [2]:
def group_and_order_filenames(filenames, maxTP, maxLevel):
    grouped_files = defaultdict(lambda: defaultdict(lambda: [None] * maxTP))
    pattern = r'^(?P<plant>[^_]+)_(?P<tube>\d+)_(?P<level>\d+)_(?P<date>\d{4}-\d{2}-\d{2})_TP(?P<timepoint>\d+)\.png$'
    #plant_tube_depth_yyyy-mm-dd_TP#

    for fname in filenames:

        match = re.match(pattern, fname)
        if match:
            plant = match.group('plant')
            tube = int(match.group('tube'))
            level = int(match.group('level'))
            date = match.group('date')
            timepoint = int(match.group('timepoint'))
            if 1 <= timepoint <= maxTP:
                grouped_files[tube][level - 1][timepoint - 1] = fname

    return grouped_files

In [3]:
def get_image_and_binary_mask(tp_file):
    img = cv2.imread("slu_data/" + tp_file)
    img_gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(img_gray)

    threshold_value = 127
    ret, img_binary = cv2.threshold(img_clahe, threshold_value, 255, cv2.THRESH_BINARY)
    img_binary[img_binary > 0] = 1
    return img, img_binary

In [4]:
def display(fname1, fname2, show=False):
    img1_full = cv2.imread(fname1)    # reference (earlier)
    img2_full = cv2.imread(fname2)      # moving (later)

    img1_gray = cv2.cvtColor(img1_full, cv2.COLOR_BGR2GRAY)
    img1_gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(img1_gray)
    img2_gray = cv2.cvtColor(img2_full, cv2.COLOR_BGR2GRAY)
    img2_gray = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(img2_gray)

    blue_tinted_image = np.zeros_like(img1_full)
    blue_tinted_image[:, :, 0] = img1_gray
    red_tinted_image = np.zeros_like(img2_full)
    red_tinted_image[:, :, 2] = img2_gray

    overlay = cv2.addWeighted(blue_tinted_image, 0.5, red_tinted_image, 0.5, 0)

    if show:
        plt.figure(figsize=(24,12))
        plt.subplot(1,1,1); plt.imshow(overlay); plt.title("Overlay after alignment")
        plt.show()

    filename_without_extension, file_extension = os.path.splitext(fname2)
    cv2.imwrite(filename_without_extension + "_correlate_overlay.png", overlay)

In [5]:
def translate_image(img, offset):
    xoffset, yoffset = offset

    # M = np.array([  [1, 0, xoffset],
    #                 [0, 1, yoffset]  ], dtype=np.float32)
    # return cv2.warpAffine(img, M, (img.shape[1], img.shape[0]))

    return shift(img, (yoffset, xoffset, 0))

In [6]:
def save_shifted(fname, aligned):
    filename_without_extension, file_extension = os.path.splitext(fname)
    cv2.imwrite("slu_data/" +filename_without_extension + "_correlate.png", aligned)

In [7]:
imgfilelist = [f for f in os.listdir("slu_data") if f.endswith(".png")]
print(f"Found {len(imgfilelist)} image files")

imgfilegroups = group_and_order_filenames(imgfilelist, 12, 7)
print(f"Found {len(imgfilelist)} image groups")

Found 5 image files
Found 5 image groups


In [8]:
for tube, depths in imgfilegroups.items():
    for depth, tp_files in depths.items():
        non_none_files = list(filter(None, tp_files))

        list_of = list(map(get_image_and_binary_mask, non_none_files))
        imgs = np.array(list(zip(*list_of))[0])
        masks = np.array(list(zip(*list_of))[1])
        shifts, result = align_masks(masks)
        shifted_full = list(map(translate_image, imgs, shifts))
        save_result = list(map(save_shifted, non_none_files, shifted_full))

        my_iterator = iter(non_none_files)
        try:
            next_item = next(my_iterator)
            while next_item:
                current_item = next_item
                next_item = next(my_iterator)
                if current_item and next_item:
                    current_without_extension, current_extension = os.path.splitext(current_item)
                    next_without_extension, next_extension = os.path.splitext(next_item)
                    if os.path.exists("slu_data/" + current_without_extension + "_correlate.png") and os.path.exists("slu_data/" + next_without_extension + "_correlate.png"):
                        display("slu_data/" + current_without_extension + "_correlate.png", "slu_data/" + next_without_extension + "_correlate.png")

        except StopIteration:
            continue